# UCC v15
它將所有的檔案（不管是文字還是影像）都視為純粹的 1D 位元組流 (Byte Stream)，只依靠演算法強大的 MatchModel 來自己發現規律。

In [2]:
!pip install cython
%load_ext Cython

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%%cython -c=-O3
# cython: language_level=3, boundscheck=False, wraparound=False, initializedcheck=False, cdivision=True

import struct
import array
import math
from libc.stdint cimport uint8_t, uint16_t, uint32_t, int32_t

cdef int _PRED_TABLE[65536]
cdef int _STRETCH[4096]
cdef int _SQUASH[8192]
cdef bint _TABLE_INIT = False

cdef inline bint is_alpha(uint8_t c):
    return (65 <= c <= 90) or (97 <= c <= 122)

cdef void init_table():
    global _TABLE_INIT
    if _TABLE_INIT: return
    cdef int n0, n1, n_total
    cdef float p
    for n0 in range(256):
        for n1 in range(256):
            n_total = n0 + n1
            if n_total == 0:
                _PRED_TABLE[(n0 << 8) | n1] = 2048
            else:
                p = (n1 + 0.5) / (n_total + 1.0)
                _PRED_TABLE[(n0 << 8) | n1] = int(p * 4095 + 0.5)
                if _PRED_TABLE[(n0 << 8) | n1] < 1:
                    _PRED_TABLE[(n0 << 8) | n1] = 1
                if _PRED_TABLE[(n0 << 8) | n1] > 4095:
                    _PRED_TABLE[(n0 << 8) | n1] = 4095

    cdef int i, val
    cdef float d
    for i in range(4096):
        if i == 0:
            _STRETCH[0] = -2048
        elif i == 4095:
            _STRETCH[4095] = 2048
        else:
            val = int(math.log(i / (4096.0 - i)) * 256.0)
            if val < -2048: val = -2048
            if val > 2048: val = 2048
            _STRETCH[i] = val

    for i in range(8192):
        d = (i - 4096) / 256.0
        val = int(4096.0 / (1.0 + math.exp(-d)))
        if val < 1: val = 1
        if val > 4095: val = 4095
        _SQUASH[i] = val

    _TABLE_INIT = True

cdef class StateMap:
    cdef public int mask
    cdef uint8_t[:] n0
    cdef uint8_t[:] n1
    def __init__(self, int bits):
        self.mask = (1 << bits) - 1
        self.n0 = bytearray(1 << bits)
        self.n1 = bytearray(1 << bits)

cdef class MatchModel:
    cdef uint8_t[:] buf
    cdef int[:] hash_table
    cdef public int buf_pos
    cdef public int buf_mask
    cdef public int hash_mask
    cdef public int match_len
    cdef public int match_pos
    cdef int depth

    def __init__(self, int buf_bits=20, int hash_bits=22, int depth=32):
        self.buf_mask = (1 << buf_bits) - 1
        self.buf = bytearray(1 << buf_bits)
        self.hash_mask = (1 << hash_bits) - 1
        self.depth = depth
        self.hash_table = array.array('i', [0]*((1 << hash_bits) * max(1, depth)))
        self.buf_pos = 0
        self.match_len = 0
        self.match_pos = 0

    cdef int predict(self, int bit_ctx):
        if self.depth == 0: return 2048
        cdef int n_bits = 0
        cdef int temp = bit_ctx
        cdef int expected_prefix, matched_prefix, bit_pos, expected_bit, conf
        if self.match_len > 0:
            while temp > 1:
                n_bits += 1
                temp >>= 1
            if n_bits > 0:
                expected_prefix = bit_ctx ^ (1 << n_bits)
                matched_prefix = self.buf[self.match_pos] >> (8 - n_bits)
                if expected_prefix != matched_prefix:
                    return 2048
            bit_pos = 7 - n_bits
            expected_bit = (self.buf[self.match_pos] >> bit_pos) & 1
            conf = 100 + self.match_len * 50
            if conf > 2000: conf = 2000
            return 2048 + conf if expected_bit else 2048 - conf
        return 2048

    cdef void update_byte(self, uint8_t new_byte, uint32_t ctx_hash):
        self.buf[self.buf_pos] = new_byte
        self.buf_pos = (self.buf_pos + 1) & self.buf_mask
        if self.depth == 0: return
        
        cdef int hash_idx = ctx_hash & self.hash_mask
        cdef int base = hash_idx * self.depth
        cdef int max_ml = 0
        cdef int best_cand = -1
        cdef int i, j, cand, ml, limit
        
        for i in range(self.depth):
            cand = self.hash_table[base + i]
            if cand == 0:
                break
            limit = self.buf_pos - cand
            if limit < 0: limit += (self.buf_mask + 1)
            if 0 < limit < (self.buf_mask + 1):
                ml = 0
                for j in range(1, 256):
                    if self.buf[(self.buf_pos - j) & self.buf_mask] == self.buf[(cand - j) & self.buf_mask]:
                        ml += 1
                    else:
                        break
                if ml > max_ml:
                    max_ml = ml
                    best_cand = cand

        if max_ml > 0:
            self.match_pos = best_cand & self.buf_mask
            self.match_len = max_ml
        else:
            self.match_len = 0

        for i in range(self.depth - 1, 0, -1):
            self.hash_table[base + i] = self.hash_table[base + i - 1]
        self.hash_table[base] = self.buf_pos

cdef class RunModel:
    cdef int last_byte
    cdef int run_count
    def __init__(self):
        self.last_byte = -1
        self.run_count = 0
    cdef int predict(self, int bit_ctx):
        cdef int n_bits = 0
        cdef int temp = bit_ctx
        cdef int decoded, expected, bit_pos, expected_bit, conf
        if self.run_count > 1:
            while temp > 1:
                n_bits += 1
                temp >>= 1
            if n_bits > 0:
                decoded = bit_ctx ^ (1 << n_bits)
                expected = (self.last_byte >> (8 - n_bits)) & ((1 << n_bits) - 1)
                if decoded != expected:
                    return 2048
            bit_pos = 7 - n_bits
            expected_bit = (self.last_byte >> bit_pos) & 1
            conf = 100 + self.run_count * 50
            if conf > 2000: conf = 2000
            return 2048 + conf if expected_bit else 2048 - conf
        return 2048
    cdef void update_byte(self, uint8_t new_byte):
        if new_byte == self.last_byte:
            if self.run_count < 255: self.run_count += 1
        else:
            self.last_byte = new_byte
            self.run_count = 1

cdef class MixerSet:
    cdef int[:] weights
    cdef int[:] last_p
    cdef int[:] last_sx
    cdef int n_models

    def __init__(self, int n_models):
        self.n_models = n_models
        self.weights = array.array('i', [16]*(8 * n_models))
        self.last_p = array.array('i', [0]*(8 * n_models))
        self.last_sx = array.array('i', [2048]*8)

    cdef int mix(self, int bit_pos, int[:] preds):
        cdef int dot = 0
        cdef int p, st
        cdef int i, idx
        for i in range(self.n_models):
            idx = bit_pos * self.n_models + i
            p = preds[i]
            self.last_p[idx] = p
            st = _STRETCH[p]
            dot += self.weights[idx] * st
        cdef int clamped_dot = dot >> 8
        if clamped_dot < -4096: clamped_dot = -4096
        if clamped_dot > 4095: clamped_dot = 4095
        cdef int sx = _SQUASH[clamped_dot + 4096]
        self.last_sx[bit_pos] = sx
        return sx

    cdef void update(self, int bit_pos, int bit):
        cdef int err = (4095 if bit else 0) - self.last_sx[bit_pos]
        cdef int w, p, st
        cdef int i, idx
        for i in range(self.n_models):
            idx = bit_pos * self.n_models + i
            w = self.weights[idx]
            p = self.last_p[idx]
            st = _STRETCH[p]
            w += (err * st + (1 << 16)) >> 17
            self.weights[idx] = w

cdef class APM:
    cdef int n_ctx
    cdef int rate
    cdef int[:] table

    def __init__(self, int n_ctx, int rate):
        self.n_ctx = n_ctx
        self.rate = rate
        self.table = array.array('i', [0]*(n_ctx * 32))
        cdef int i, j
        for i in range(n_ctx):
            for j in range(32):
                self.table[i * 32 + j] = (j * 4095) // 31

    cdef int predict(self, int ctx, int p):
        ctx = ctx % self.n_ctx
        cdef int x = p * 31
        cdef int bin_idx = x >> 12
        cdef int base_idx = ctx * 32
        if bin_idx >= 31: return self.table[base_idx + 31]
        cdef int frac = x & 0xFFF
        cdef int t0 = self.table[base_idx + bin_idx]
        cdef int t1 = self.table[base_idx + bin_idx + 1]
        cdef int p_apm = t0 + (((t1 - t0) * frac + 2048) >> 12)
        if p_apm < 1: return 1
        if p_apm > 4095: return 4095
        return p_apm

    cdef void update(self, int ctx, int p, int bit):
        ctx = ctx % self.n_ctx
        cdef int x = p * 31
        cdef int bin_idx = x >> 12
        cdef int target = 4095 if bit else 0
        cdef int err
        cdef int base_idx = ctx * 32
        if bin_idx < 32:
            err = target - self.table[base_idx + bin_idx]
            self.table[base_idx + bin_idx] += (err + (1 << (self.rate - 1))) >> self.rate
        if bin_idx + 1 < 32:
            err = target - self.table[base_idx + bin_idx + 1]
            self.table[base_idx + bin_idx + 1] += (err + (1 << (self.rate - 1))) >> self.rate

cdef class BitEncoder:
    cdef bytearray out
    cdef uint32_t lo
    cdef uint32_t hi
    def __init__(self):
        self.out = bytearray()
        self.lo = 0
        self.hi = 0xFFFFFFFF
    cdef void encode(self, int bit, int p):
        cdef uint32_t mid = self.lo + ((self.hi - self.lo) >> 12) * (4096 - p)
        if bit: self.lo = mid + 1
        else: self.hi = mid
        while ((self.lo ^ self.hi) & 0xFF000000) == 0:
            self.out.append(self.hi >> 24)
            self.lo = (self.lo << 8) & 0xFFFFFFFF
            self.hi = ((self.hi << 8) | 0xFF) & 0xFFFFFFFF
    def flush(self):
        self.out.append(self.hi >> 24)
        self.out.append((self.hi >> 16) & 0xFF)
        self.out.append((self.hi >> 8) & 0xFF)
        self.out.append(self.hi & 0xFF)
        return bytes(self.out)

cdef class BitDecoder:
    cdef bytes data
    cdef int pos
    cdef uint32_t lo
    cdef uint32_t hi
    cdef uint32_t code
    cdef int length
    def __init__(self, bytes data):
        self.data = data
        self.length = len(data)
        self.pos = 0
        self.lo = 0
        self.hi = 0xFFFFFFFF
        self.code = 0
        cdef int i
        for i in range(4): self.code = (self.code << 8) | self._byte()
    cdef int _byte(self):
        if self.pos < self.length:
            b = self.data[self.pos]
            self.pos += 1
            return b
        return 0
    cdef int decode(self, int p):
        cdef uint32_t mid = self.lo + ((self.hi - self.lo) >> 12) * (4096 - p)
        cdef int bit = 1 if self.code > mid else 0
        if bit: self.lo = mid + 1
        else: self.hi = mid
        while ((self.lo ^ self.hi) & 0xFF000000) == 0:
            self.lo = (self.lo << 8) & 0xFFFFFFFF
            self.hi = ((self.hi << 8) | 0xFF) & 0xFFFFFFFF
            self.code = ((self.code << 8) | self._byte()) & 0xFFFFFFFF
        return bit

cdef class ContextModel:
    cdef list sm
    cdef list sparse_sm
    cdef uint32_t[:] sparse_hash
    cdef MatchModel match4
    cdef MatchModel match3
    cdef RunModel run_model
    cdef StateMap word_sm
    cdef uint32_t word_hash
    cdef StateMap indirect_sm
    cdef uint32_t[:] indirect_table
    cdef MixerSet mixer_set
    cdef APM apm1
    cdef APM apm2
    cdef APM apm3
    cdef uint32_t[:] ctx_hash
    cdef list history
    cdef int bit_ctx
    cdef uint32_t[:] base_hash
    cdef int byte_count
    cdef int[:] preds

    def __init__(self):
        init_table()
        
        self.sm = [StateMap(16), StateMap(20), StateMap(22), StateMap(22),
                   StateMap(21), StateMap(20), StateMap(19), StateMap(19)]
        
        self.sparse_sm = [StateMap(21) for _ in range(6)]
        self.sparse_hash = array.array('I', [0]*6)
        
        self.match4 = MatchModel(20, 22, 32)
        self.match3 = MatchModel(20, 22, 32)
        self.word_sm = StateMap(21)
        self.word_hash = 0
        self.indirect_sm = StateMap(22)
        self.indirect_table = array.array('I', [0]*(1<<20))
        
        self.run_model = RunModel()
        
        self.mixer_set = MixerSet(19)
        self.preds = array.array('i', [0]*19)
        
        self.apm1 = APM(256, 4)
        self.apm2 = APM(4096, 5)
        self.apm3 = APM(32768, 6)
        self.ctx_hash = array.array('I', [0]*8)
        self.history = []
        self.bit_ctx = 1
        self.base_hash = array.array('I', [0]*8)
        self.byte_count = 0

    cdef int predict(self):
        cdef int bc = self.bit_ctx
        cdef int n = len(self.history)
        cdef int i, idx
        cdef uint32_t sh
        cdef StateMap s
        cdef int n_bits = 0
        cdef int temp = bc
        cdef int mixed, p1, p2, p3, final_p
        cdef int prev, prev2, ctx2, ctx3
        cdef uint32_t ctx2_masked = 0
        cdef uint32_t indirect_state = 0

        for i in range(8):
            self.ctx_hash[i] = ((self.base_hash[i] * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & 0xFFFFFFFF
            s = self.sm[i]
            idx = self.ctx_hash[i] & s.mask
            self.preds[i] = _PRED_TABLE[(s.n0[idx] << 8) | s.n1[idx]]

        if n >= 3:
            sh = ((self.history[n-1] * 0x85EBCA6B) & 0xFFFFFFFF)
            sh = ((sh * 0x9E3779B1) ^ (self.history[n-3] * 0x85EBCA6B)) & 0xFFFFFFFF
            self.sparse_hash[0] = ((sh * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & 0xFFFFFFFF
        else: self.sparse_hash[0] = (bc * 0x85EBCA6B) & 0xFFFFFFFF
        
        if n >= 4:
            sh = ((self.history[n-1] * 0x85EBCA6B) & 0xFFFFFFFF)
            sh = ((sh * 0x9E3779B1) ^ (self.history[n-4] * 0x85EBCA6B)) & 0xFFFFFFFF
            self.sparse_hash[1] = ((sh * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & 0xFFFFFFFF
        else: self.sparse_hash[1] = ((bc * 0xCC9E2D51) + 0x1B873593) & 0xFFFFFFFF
        
        if n >= 6:
            sh = ((self.history[n-1] * 0x85EBCA6B) & 0xFFFFFFFF)
            sh = ((sh * 0x9E3779B1) ^ (self.history[n-5] * 0x85EBCA6B)) & 0xFFFFFFFF
            sh = ((sh * 0x9E3779B1) ^ (self.history[n-6] * 0x85EBCA6B)) & 0xFFFFFFFF
            self.sparse_hash[2] = ((sh * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & 0xFFFFFFFF
        else: self.sparse_hash[2] = ((bc * 0x9E3779B1) + 0x12345678) & 0xFFFFFFFF
        
        if n >= 4:
            sh = ((self.history[n-2] * 0x85EBCA6B) & 0xFFFFFFFF)
            sh = ((sh * 0x9E3779B1) ^ (self.history[n-3] * 0x85EBCA6B)) & 0xFFFFFFFF
            self.sparse_hash[3] = ((sh * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & 0xFFFFFFFF
        else: self.sparse_hash[3] = ((bc * 0x1B873593) + 0x85EBCA6B) & 0xFFFFFFFF
        
        if n >= 5:
            sh = ((self.history[n-2] * 0x85EBCA6B) & 0xFFFFFFFF)
            sh = ((sh * 0x9E3779B1) ^ (self.history[n-4] * 0x85EBCA6B)) & 0xFFFFFFFF
            self.sparse_hash[4] = ((sh * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & 0xFFFFFFFF
        else: self.sparse_hash[4] = ((bc * 0x85EBCA6B) + 0xCC9E2D51) & 0xFFFFFFFF
        
        if n >= 5:
            sh = ((self.history[n-3] * 0x85EBCA6B) & 0xFFFFFFFF)
            sh = ((sh * 0x9E3779B1) ^ (self.history[n-4] * 0x85EBCA6B)) & 0xFFFFFFFF
            self.sparse_hash[5] = ((sh * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & 0xFFFFFFFF
        else: self.sparse_hash[5] = ((bc * 0x9E3779B1) + 0x85EBCA6B) & 0xFFFFFFFF

        for i in range(6):
            s = self.sparse_sm[i]
            idx = self.sparse_hash[i] & s.mask
            self.preds[8 + i] = _PRED_TABLE[(s.n0[idx] << 8) | s.n1[idx]]

        self.preds[14] = self.match4.predict(bc)
        self.preds[15] = self.match3.predict(bc)
        self.preds[16] = self.run_model.predict(bc)

        idx = ((self.word_hash * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & self.word_sm.mask
        self.preds[17] = _PRED_TABLE[(self.word_sm.n0[idx] << 8) | self.word_sm.n1[idx]]
        
        ctx2_masked = self.base_hash[2] & 0xFFFFF
        indirect_state = self.indirect_table[ctx2_masked]
        idx = ((indirect_state * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & self.indirect_sm.mask
        self.preds[18] = _PRED_TABLE[(self.indirect_sm.n0[idx] << 8) | self.indirect_sm.n1[idx]]

        while temp > 1:
            n_bits += 1
            temp >>= 1

        mixed = self.mixer_set.mix(n_bits, self.preds)
        p1 = self.apm1.predict(bc & 0xFF, mixed)

        prev = self.history[n-1] if n > 0 else 0
        prev2 = self.history[n-2] if n > 1 else 0

        ctx2 = ((prev >> 4) << 8) | (bc & 0xFF)
        ctx3 = ((prev2 >> 5) << 12) | ((prev >> 4) << 8) | (bc & 0xFF)

        p2 = self.apm2.predict(ctx2, p1)
        p3 = self.apm3.predict(ctx3, p2)

        final_p = (p3 * 3 + mixed) >> 2

        if final_p < 1: final_p = 1
        if final_p > 4095: final_p = 4095
        return final_p

    cdef void update(self, int bit):
        cdef int bc = self.bit_ctx
        cdef int n_bits = 0
        cdef int temp = bc
        cdef int i, idx, n0v, n1v
        cdef StateMap s
        cdef int mixed, p1, p2, n, j, k
        cdef uint8_t new_byte
        cdef uint32_t ctx3_hash = 0
        cdef uint32_t ctx4_hash = 0
        cdef uint32_t h
        cdef int prev, prev2, ctx2, ctx3
        cdef uint32_t ctx2_masked = 0
        cdef uint32_t indirect_state = 0

        while temp > 1:
            n_bits += 1
            temp >>= 1

        for i in range(8):
            s = self.sm[i]
            idx = self.ctx_hash[i] & s.mask
            if bit:
                n1v = s.n1[idx] + 1
                if n1v > 250:
                    s.n0[idx] = (s.n0[idx] + 1) >> 1
                    s.n1[idx] = (n1v + 1) >> 1
                else: s.n1[idx] = n1v
            else:
                n0v = s.n0[idx] + 1
                if n0v > 250:
                    s.n0[idx] = (n0v + 1) >> 1
                    s.n1[idx] = (s.n1[idx] + 1) >> 1
                else: s.n0[idx] = n0v

        for i in range(6):
            s = self.sparse_sm[i]
            idx = self.sparse_hash[i] & s.mask
            if bit:
                n1v = s.n1[idx] + 1
                if n1v > 250:
                    s.n0[idx] = (s.n0[idx] + 1) >> 1
                    s.n1[idx] = (n1v + 1) >> 1
                else: s.n1[idx] = n1v
            else:
                n0v = s.n0[idx] + 1
                if n0v > 250:
                    s.n0[idx] = (n0v + 1) >> 1
                    s.n1[idx] = (s.n1[idx] + 1) >> 1
                else: s.n0[idx] = n0v

        idx = ((self.word_hash * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & self.word_sm.mask
        s = self.word_sm
        if bit:
            n1v = s.n1[idx] + 1
            if n1v > 250:
                s.n0[idx] = (s.n0[idx] + 1) >> 1
                s.n1[idx] = (n1v + 1) >> 1
            else: s.n1[idx] = n1v
        else:
            n0v = s.n0[idx] + 1
            if n0v > 250:
                s.n0[idx] = (n0v + 1) >> 1
                s.n1[idx] = (s.n1[idx] + 1) >> 1
            else: s.n0[idx] = n0v
                
        ctx2_masked = self.base_hash[2] & 0xFFFFF
        indirect_state = self.indirect_table[ctx2_masked]
        idx = ((indirect_state * 0x9E3779B1) ^ (bc * 0x85EBCA6B)) & self.indirect_sm.mask
        s = self.indirect_sm
        if bit:
            n1v = s.n1[idx] + 1
            if n1v > 250:
                s.n0[idx] = (s.n0[idx] + 1) >> 1
                s.n1[idx] = (n1v + 1) >> 1
            else: s.n1[idx] = n1v
        else:
            n0v = s.n0[idx] + 1
            if n0v > 250:
                s.n0[idx] = (n0v + 1) >> 1
                s.n1[idx] = (s.n1[idx] + 1) >> 1
            else: s.n0[idx] = n0v

        self.mixer_set.update(n_bits, bit)
        mixed = self.mixer_set.last_sx[n_bits]

        self.apm1.update(bc & 0xFF, mixed, bit)
        p1 = self.apm1.predict(bc & 0xFF, mixed)

        n = len(self.history)
        prev = self.history[n-1] if n > 0 else 0
        prev2 = self.history[n-2] if n > 1 else 0

        ctx2 = ((prev >> 4) << 8) | (bc & 0xFF)
        ctx3 = ((prev2 >> 5) << 12) | ((prev >> 4) << 8) | (bc & 0xFF)

        self.apm2.update(ctx2, p1, bit)
        p2 = self.apm2.predict(ctx2, p1)
        self.apm3.update(ctx3, p2, bit)

        bc = (bc << 1) | bit
        self.bit_ctx = bc

        if bc >= 256:
            new_byte = bc & 0xFF
            self.history.append(new_byte)
            n = len(self.history)
            
            if is_alpha(new_byte):
                self.word_hash = (self.word_hash * 33 + new_byte) & 0xFFFFFFFF
            else:
                self.word_hash = 0
            self.indirect_table[ctx2_masked] = new_byte

            for j in range(max(0, n - 4), n):
                ctx4_hash = ((ctx4_hash * 0x9E3779B1) ^ (self.history[j] * 0x85EBCA6B)) & 0xFFFFFFFF
            self.match4.update_byte(new_byte, ctx4_hash)
            
            for j in range(max(0, n - 3), n):
                ctx3_hash = ((ctx3_hash * 0x9E3779B1) ^ (self.history[j] * 0x85EBCA6B)) & 0xFFFFFFFF
            self.match3.update_byte(new_byte, ctx3_hash)
            
            self.run_model.update_byte(new_byte)

            if n > 16:
                del self.history[0]
                n -= 1
            self.bit_ctx = 1
            self.byte_count += 1

            self.base_hash[0] = 0
            if n >= 1:
                self.base_hash[1] = (self.history[n-1] * 0x85EBCA6B) & 0xFFFFFFFF
            if n >= 2:
                self.base_hash[2] = (((self.history[n-2] * 0x85EBCA6B) & 0xFFFFFFFF) * 0x9E3779B1 ^ (self.history[n-1] * 0x85EBCA6B)) & 0xFFFFFFFF
            if n >= 3:
                h = (self.history[n-3] * 0x85EBCA6B) & 0xFFFFFFFF
                h = ((h * 0x9E3779B1) ^ (self.history[n-2] * 0x85EBCA6B)) & 0xFFFFFFFF
                self.base_hash[3] = ((h * 0x9E3779B1) ^ (self.history[n-1] * 0x85EBCA6B)) & 0xFFFFFFFF
            if n >= 4:
                h = (self.history[n-4] * 0x85EBCA6B) & 0xFFFFFFFF
                h = ((h * 0x9E3779B1) ^ (self.history[n-3] * 0x85EBCA6B)) & 0xFFFFFFFF
                h = ((h * 0x9E3779B1) ^ (self.history[n-2] * 0x85EBCA6B)) & 0xFFFFFFFF
                self.base_hash[4] = ((h * 0x9E3779B1) ^ (self.history[n-1] * 0x85EBCA6B)) & 0xFFFFFFFF
            if n >= 6:
                h = (self.history[n-6] * 0x85EBCA6B) & 0xFFFFFFFF
                for k in range(-5, 0):
                    h = ((h * 0x9E3779B1) ^ (self.history[n+k] * 0x85EBCA6B)) & 0xFFFFFFFF
                self.base_hash[5] = h
            if n >= 8:
                h = (self.history[n-8] * 0x85EBCA6B) & 0xFFFFFFFF
                for k in range(-7, 0):
                    h = ((h * 0x9E3779B1) ^ (self.history[n+k] * 0x85EBCA6B)) & 0xFFFFFFFF
                self.base_hash[6] = h
            if n >= 12:
                h = (self.history[n-12] * 0x85EBCA6B) & 0xFFFFFFFF
                for k in range(-11, 0):
                    h = ((h * 0x9E3779B1) ^ (self.history[n+k] * 0x85EBCA6B)) & 0xFFFFFFFF
                self.base_hash[7] = h

def compress(bytes data):
    cdef int total = len(data)
    cdef bytearray result = bytearray(struct.pack("<I", total))
    cdef BitEncoder enc
    cdef ContextModel cm
    cdef int bit_pos, p, bit
    cdef uint8_t byte

    if total == 0: return bytes(result)

    enc = BitEncoder()
    cm = ContextModel()
    for byte in data:
        for bit_pos in range(7, -1, -1):
            bit = (byte >> bit_pos) & 1
            p = cm.predict()
            enc.encode(bit, p)
            cm.update(bit)
    result.extend(enc.flush())
    return bytes(result)

def decompress(bytes compressed):
    cdef int original_size
    cdef BitDecoder dec
    cdef ContextModel cm
    cdef bytearray output
    cdef int i, bit_pos, p, bit, byte

    if len(compressed) < 4: raise ValueError("Data format error")
    original_size = struct.unpack("<I", compressed[:4])[0]
    if original_size == 0: return b""

    dec = BitDecoder(compressed[4:])
    cm = ContextModel()
    output = bytearray()
    for i in range(original_size):
        byte = 0
        for bit_pos in range(7, -1, -1):
            p = cm.predict()
            bit = dec.decode(p)
            cm.update(bit)
            byte = (byte << 1) | bit
        output.append(byte)
    return bytes(output)


Content of stdout:
_cython_magic_5b1ef2d18ab25b641f84ac7f6d29b5e17fd3c928c55c712c1d1263a701d4b2b6.c
C:\Users\Leader\.ipython\cython\_cython_magic_5b1ef2d18ab25b641f84ac7f6d29b5e17fd3c928c55c712c1d1263a701d4b2b6.c(16523): warning C4244: '=': ±N 'double' Âà´«¬° 'float'¡A¥Ñ©óÃþ«¬¤£¦P¡A¥i¯à¾É­P¸ê®Æ¿ò¥¢
C:\Users\Leader\.ipython\cython\_cython_magic_5b1ef2d18ab25b641f84ac7f6d29b5e17fd3c928c55c712c1d1263a701d4b2b6.c(16751): warning C4244: '=': ±N 'double' Âà´«¬° 'float'¡A¥Ñ©óÃþ«¬¤£¦P¡A¥i¯à¾É­P¸ê®Æ¿ò¥¢
C:\Users\Leader\.ipython\cython\_cython_magic_5b1ef2d18ab25b641f84ac7f6d29b5e17fd3c928c55c712c1d1263a701d4b2b6.c(23006): warning C4244: '=': ±N 'Py_ssize_t' Âà´«¬° 'int'¡A¥Ñ©óÃþ«¬¤£¦P¡A¥i¯à¾É­P¸ê®Æ¿ò¥¢
C:\Users\Leader\.ipython\cython\_cython_magic_5b1ef2d18ab25b641f84ac7f6d29b5e17fd3c928c55c712c1d1263a701d4b2b6.c(24555): warning C4244: '=': ±N 'Py_ssize_t' Âà´«¬° 'int'¡A¥Ñ©óÃþ«¬¤£¦P¡A¥i¯à¾É­P¸ê®Æ¿ò¥¢
C:\Users\Leader\.ipython\cython\_cython_magic_5b1ef2d18ab25b641f84ac7f6d29b5e17fd3c928c55c712c1d

In [4]:
import os
import time

def run_benchmark(filepaths):
    total_compress = 0
    total_decompress = 0
    print("┌─────────────────────────────────────────────────────────────────────────┐")
    print("│ 檔案                 原始        壓縮後    壓縮倍率   節省% 驗證              │")
    print("├─────────────────────────────────────────────────────────────────────────┤")

    for filepath in filepaths:
        if not os.path.exists(filepath):
            continue

        with open(filepath, 'rb') as f:
            raw = f.read()

        start = time.time()
        core_comp = compress(raw)
        comp_time = time.time() - start

        start = time.time()
        decomp = decompress(core_comp)
        
        is_valid = (decomp == raw)

        decomp_time = time.time() - start

        real_comp_size = len(core_comp)
        ratio = real_comp_size / len(raw) if len(raw) > 0 else 1
        multiplier = len(raw) / real_comp_size if real_comp_size > 0 else float('inf')
        saving = (1 - ratio) * 100
        valid_str = "✓ OK" if is_valid else "✗ FAIL"

        print(f"│ {filepath:<15} {len(raw):>10,} {real_comp_size:>10,}   {multiplier:>5.2f}x   {saving:>4.1f}% {valid_str} │")

        total_compress += comp_time
        total_decompress += decomp_time

    print("└─────────────────────────────────────────────────────────────────────────┘")
    print(f"\n總壓縮時間: {total_compress:.2f} 秒, 總解壓時間: {total_decompress:.2f} 秒")

run_benchmark(['test1.txt', 'test2.txt', 'test3.txt', 'Cameraman.bmp', 'Lenna.bmp'])


┌─────────────────────────────────────────────────────────────────────────┐
│ 檔案                 原始        壓縮後    壓縮倍率   節省% 驗證              │
├─────────────────────────────────────────────────────────────────────────┤
│ test1.txt               35         33    1.06x    5.7% ✓ OK │
│ test2.txt            2,638      1,023    2.58x   61.2% ✓ OK │
│ test3.txt            5,349      2,019    2.65x   62.3% ✓ OK │
│ Cameraman.bmp       66,616     38,530    1.73x   42.2% ✓ OK │
│ Lenna.bmp          263,224    169,675    1.55x   35.5% ✓ OK │
└─────────────────────────────────────────────────────────────────────────┘

總壓縮時間: 33.32 秒, 總解壓時間: 31.07 秒


## 若老師想要做額外的測資修改下面的，但上面的一樣要跑過~

In [ ]:
run_benchmark(["test1.txt"]) # 假設我要跑 test1.txt